# Análisis de Estabilidad Temporal y Regional
## Tesis: Benchmarking Explainable Gradient Boosting and Tabular Deep Learning
## for Predicting Satisfaction with Democracy in Latin America (1995–2024)

**Objetivo:** Responder PI3 y OE3 — ¿los determinantes identificados son
estables entre splits históricos y subregiones geográficas?

Requiere haber ejecutado los notebooks 02 y 04 previamente.

### Estructura
| Sección | Contenido |
|---|---|
| 1–2 | Importaciones y configuración |
| 3 | Correlación Spearman entre rankings SHAP (test/test/test) |
| 4 | Heatmap y bump chart de estabilidad |
| 5 | Análisis de estabilidad por subregión |
| 6 | MAE ordinal por país y split |
| 7 | Prueba formal de H4 y H5 |
| 8 | Resumen y guardado |

## 1. Importaciones

In [ ]:
import sys
sys.path.append("..")

import warnings
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import json

warnings.filterwarnings("ignore")

from utils.config import (
    setup_plots, THEME, PATHS, SPLIT, SUBREGIONES, PAISES_EXCLUIR_EVAL,
    PAISES_EXCLUIR_EVAL, COL_TARGET, COL_PAIS,
    ETIQUETAS, ETIQUETAS_FEATURES, BLOQUES, bloque_de,
)
from utils.io import (
    cargar_pipeline, cargar_split_parquet, cargar_resultados,
    cargar_shap_values, shap_disponible, cargar_mejor_modelo,
)
from utils.plots import (
    plot_heatmap_estabilidad, plot_bump_chart,
    plot_spearman_estabilidad, plot_shap_por_subregion,
    plot_rendimiento_por_pais, save_figure, model_color,
)
from utils.metrics import evaluar
from sklearn.metrics import mean_absolute_error

setup_plots()
print("✓ Importaciones completadas.")

## 2. Configuración

In [ ]:
# =============================================================================
# Parámetros — NB05: Análisis por subregiones y estrategias de balanceo
# Propósito rediseñado:
#   - Análisis de rendimiento por subregión (responde a solicitud del tutor)
#   - Análisis de SHAP por subregión (¿qué explica la satisfacción por región?)
#   - Comparativa SMOTE-NC vs. pesos de clase en clases minoritarias
# =============================================================================
from utils.io import cargar_pipeline, cargar_resultados, cargar_shap_values, cargar_split_parquet
from sklearn.metrics import mean_absolute_error, cohen_kappa_score, accuracy_score

MODELO_PRINCIPAL = "CatBoost"    # actualizar tras ver resultados de NB03
ESTRATEGIA_PPAL  = "pesos_clase" # mejor estrategia de E1

setup_plots()
print(f"Modelo principal  : {MODELO_PRINCIPAL}")
print(f"Estrategia ppal   : {ESTRATEGIA_PPAL}")
print(f"Test set          : {SPLIT['test']}")
print(f"Países en test    : 16 (sin Venezuela ni Nicaragua)")
print(f"Subregiones       : {list(SUBREGIONES.keys())}")


## 3. Correlación Spearman entre rankings SHAP

La correlación de Spearman entre los rankings de importancia SHAP de
distintos splits mide directamente la estabilidad de los determinantes.

- **r = 1.0**: los mismos factores explican la satisfacción en todos los períodos
- **r < 0.7**: cambio sustancial en los determinantes entre períodos
- **r < 0.5**: cambio estructural (umbral para H5)

In [ ]:
# =============================================================================
# Cargar resultados y datos del split de prueba
# =============================================================================
df_res = cargar_resultados(split="test")
df_te  = cargar_split_parquet("test")

# Cargar artefacto del modelo principal
art = cargar_pipeline(MODELO_PRINCIPAL)
feats = art["features"]

# Predicciones del modelo principal
from utils.preprocessing import construir_split
X_tr, y_tr, X_val, y_val, X_te, y_te, w_tr, w_val, w_te = construir_split(
    df_te.assign(**{c: df_te[c] for c in feats if c in df_te.columns}),
    feats, {0:1,1:1,2:1,3:1}
)

from utils.config import COL_PAIS, COL_TARGET
print(f"Test set cargado: {len(df_te):,} registros")
print(f"Países: {sorted(df_te[COL_PAIS].unique()) if COL_PAIS in df_te.columns else 'N/A'}")


## 4. Visualizaciones de estabilidad temporal

In [ ]:
# =============================================================================
# Análisis de rendimiento por subregión — modelo principal
# =============================================================================
if COL_PAIS not in df_te.columns:
    print("⚠ Columna pais_nombre no disponible en el test parquet")
else:
    from utils.preprocessing import construir_split
    import numpy as np

    # Predicciones sobre el test completo
    if art["tipo_modelo"] in ("olo","tabnet"):
        vars_cat = [c for c in art.get("vars_categoricas",[]) if c in df_te.columns]
        cols_num = [c for c in feats if c not in vars_cat]
        X_num = pd.DataFrame(art["imp_num"].transform(df_te[feats][cols_num]),
                             columns=cols_num, index=df_te.index)
        if vars_cat and art.get("imp_cat"):
            X_cat = pd.DataFrame(art["imp_cat"].transform(df_te[feats][vars_cat]),
                                 columns=vars_cat, index=df_te.index)
            X_imp = pd.concat([X_num, X_cat], axis=1)[feats]
        else:
            X_imp = X_num
        cols_sc = [c for c in feats if c not in vars_cat]
        X_sc = X_imp.copy()
        X_sc[cols_sc] = art["scaler"].transform(X_imp[cols_sc])
        y_pred_all = art["modelo"].predict(X_sc.values)
    else:
        y_pred_all = art["modelo"].predict(df_te[feats])
        if hasattr(y_pred_all, "flatten"):
            y_pred_all = y_pred_all.flatten()
    y_pred_all = np.array(y_pred_all).astype(int)
    y_true_all = df_te[COL_TARGET].values.astype(int)

    filas_sr = []
    for sr_name, paises_sr in SUBREGIONES.items():
        mask = df_te[COL_PAIS].isin(paises_sr)
        if mask.sum() == 0:
            continue
        y_t = y_true_all[mask.values]
        y_p = y_pred_all[mask.values]
        mae  = mean_absolute_error(y_t, y_p)
        kappa = cohen_kappa_score(y_t, y_p, weights="quadratic") if len(set(y_t))>1 else float("nan")
        acc  = accuracy_score(y_t, y_p)
        mejor_pais = None; peor_pais = None
        min_mae = 999; max_mae = -1
        for p in paises_sr:
            mp = df_te[COL_PAIS] == p
            if mp.sum() < 5:
                continue
            mae_p = mean_absolute_error(y_true_all[mp.values], y_pred_all[mp.values])
            if mae_p < min_mae: min_mae = mae_p; mejor_pais = p
            if mae_p > max_mae: max_mae = mae_p; peor_pais  = p
        filas_sr.append({"subregion":sr_name, "n_registros":mask.sum(),
                          "mae_ordinal":round(mae,4), "kappa_cuadratico":round(kappa,4),
                          "accuracy":round(acc,4), "mejor_pais":mejor_pais,
                          "peor_pais":peor_pais})
        print(f"  {sr_name:<22}: MAE={mae:.4f}  κ={kappa:.4f}  Acc={acc:.4f}")

    df_sr = pd.DataFrame(filas_sr)
    df_sr.to_csv(PATHS["FOLDER_RESULTS_TABLES"] / "mae_subregiones.csv", index=False)
    print()
    print("✓ Guardado: results/tables/mae_subregiones.csv")
    print("  → Tabla 5.9 de la tesis: MAE por subregión")


In [ ]:
# =============================================================================
# SHAP medio por subregión — top 5 variables por región
# =============================================================================
try:
    df_shap = pd.read_csv(PATHS["FOLDER_RESULTS_TABLES"] / f"shap_importancias_{MODELO_PRINCIPAL}.csv")
    print("Top 5 variables SHAP por subregión:")
    for sr_name, paises_sr in SUBREGIONES.items():
        mask = df_te[COL_PAIS].isin(paises_sr)
        if mask.sum() < 20:
            continue
        shap_sr = df_shap.loc[mask.values[:len(df_shap)]].mean().sort_values(ascending=False)
        top5 = shap_sr.head(5).index.tolist()
        etiq = [ETIQUETAS_FEATURES.get(v,v) for v in top5]
        print(f"  {sr_name:<22}: {etiq}")
except FileNotFoundError:
    print("⚠ Ejecutar NB04 primero para calcular SHAP")


## 5. Análisis de estabilidad por subregión

In [ ]:
# =============================================================================
# Comparativa E1: Rendimiento en clases minoritarias por estrategia
# ¿SMOTE-NC mejora el F1 de la clase 0 (Para nada satisfecho)?
# =============================================================================
df_test_res = cargar_resultados(split="test")
df_ord = df_test_res[df_test_res["variante_target"]=="ordinal_4clases"].copy()

if not df_ord.empty:
    from sklearn.metrics import classification_report
    print("F1 de clase 0 (Para nada satisfecho) por estrategia:")
    print("(Calculado sobre las predicciones del conjunto de prueba)")
    print()
    for est in ["sin_balanceo","pesos_clase","smotenc"]:
        sub = df_ord[df_ord["estrategia_balanceo"]==est]
        if sub.empty:
            print(f"  {est}: sin resultados")
            continue
        mejor = sub.loc[sub["kappa_cuadratico"].idxmax()]
        print(f"  {est:<15}: Kappa={mejor['kappa_cuadratico']:.4f}  "
              f"F1_macro={mejor['f1_macro']:.4f}  Acc={mejor['accuracy']:.4f}")

    df_ord.to_csv(PATHS["FOLDER_RESULTS_TABLES"] / "comparativa_estrategias_balanceo.csv", index=False)
    print()
    print("✓ Guardado: results/tables/comparativa_estrategias_balanceo.csv")
    print("  → Tabla 5.X de la tesis: Comparativa E1")
else:
    print("⚠ Sin resultados de E1 — ejecutar NB02")


## 6. MAE ordinal por país y split

In [ ]:
# =============================================================================
# Análisis de rendimiento por país para los 3 splits
# Métrica: MAE ordinal (robusta ante clases minoritarias)
# =============================================================================
import joblib

filas_mae = []

for sp in ["test","test","test"]:
    df_te = cargar_split_parquet(sp, "test")
    if COL_PAIS not in df_te.columns:
        continue

    y_te = df_te[COL_TARGET].astype(int).values

    for modelo in ["OLO","XGBoost","CatBoost","LightGBM","TabNet"]:
        try:
            art  = cargar_pipeline(modelo, sp)
            tipo = art["tipo_modelo"]
            feats_m = art["features"]
            X_te = df_te[[f for f in feats_m if f in df_te.columns]]
            X_te = X_te.reindex(columns=feats_m)

            if tipo in ("olo", "tabnet"):
                vars_cat_m = [c for c in art.get("vars_categoricas", []) if c in X_te.columns]
                cols_num_m = [c for c in feats_m if c not in vars_cat_m]
                X_num_m = pd.DataFrame(art["imp_num"].transform(X_te[cols_num_m]),
                                    columns=cols_num_m, index=X_te.index)
                if vars_cat_m and art.get("imp_cat") is not None:
                    X_cat_m = pd.DataFrame(art["imp_cat"].transform(X_te[vars_cat_m]),
                                        columns=vars_cat_m, index=X_te.index)
                    X_imp = pd.concat([X_num_m, X_cat_m], axis=1)[feats_m]
                else:
                    X_imp = X_num_m
                cols_sc_m = [c for c in feats_m if c not in vars_cat_m]
                X_sc = X_imp.copy()
                X_sc[cols_sc_m] = art["scaler"].transform(X_imp[cols_sc_m])
                y_raw = art["modelo"].predict(X_sc.values if tipo == "olo"
                            else X_sc.values.astype(np.float32))
            else:
                X_in = X_te.copy()
                if modelo == "CatBoost":
                    for col in art.get("vars_categoricas",[]):
                        if col in X_in.columns:
                            X_in[col] = X_in[col].fillna(-999).astype(int).astype(str)
                y_raw = art["modelo"].predict(X_in)

            y_pred = y_raw.flatten() if hasattr(y_raw,"flatten") else y_raw

            # MAE por país
            for pais in df_te[COL_PAIS].unique():
                if pais in PAISES_EXCLUIR_TEST:
                    continue
                mask = df_te[COL_PAIS] == pais
                if mask.sum() < 10:
                    continue
                mae = mean_absolute_error(y_te[mask], y_pred[mask])
                sr  = next((s for s, ps in SUBREGIONES.items() if pais in ps),
                           "Sin clasificar")
                filas_mae.append({
                    "modelo":modelo,"split":sp,"pais":pais,
                    "subregion":sr,"mae_ordinal":mae,"n":int(mask.sum()),
                })
        except FileNotFoundError:
            pass
        except Exception as e:
            print(f"  ⚠ {modelo} {sp}: {e}")

df_mae_completo = pd.DataFrame(filas_mae)
if not df_mae_completo.empty:
    print("MAE Ordinal promedio por modelo y split:")
    pivot_mae = df_mae_completo.groupby(["modelo","split"])["mae_ordinal"].mean().unstack()
    print(pivot_mae.round(4).to_string())
    df_mae_completo.to_csv(
        PATHS["FOLDER_RESULTS_TABLES"] / "mae_por_pais_todos.csv", index=False)
    print("\n✓ Tabla MAE completa guardada")

In [ ]:
# Heatmap: MAE por subregión × split para el modelo principal
if not df_mae_completo.empty:
    df_sr_sp = (df_mae_completo[df_mae_completo["modelo"] == MODELO_ESTABILIDAD]
                .groupby(["subregion","split"])["mae_ordinal"].mean()
                .unstack())

    fig, ax = plt.subplots(figsize=(8, 5))
    sns.heatmap(df_sr_sp, annot=True, fmt=".3f", cmap="YlOrRd",
                linewidths=0.3, ax=ax,
                cbar_kws={"label": "MAE Ordinal"})
    ax.set_title(f"MAE Ordinal por subregión y split — {MODELO_ESTABILIDAD}",
                 fontweight="bold")
    save_figure(f"05_mae_subregion_split_{MODELO_ESTABILIDAD}")
    plt.show()

## 7. Prueba formal de H4 y H5

**H4:** la importancia de las variables varía significativamente entre
splits, con seguridad ciudadana y corrupción ganando peso en el split único.

**H5:** la correlación Spearman test-test será significativamente menor que test-test,
evidenciando cambio estructural post-pandemia.

In [ ]:
# =============================================================================
# Prueba de H4: ¿Qué bloques cambiaron más entre el split único?
# =============================================================================
if 'df_cambio' in dir() and not df_cambio.empty:
    print("=" * 60)
    print("PRUEBA DE H4")
    print("=" * 60)
    print("Cambio en importancia SHAP por bloque (test → test):")
    print()

    pred_h4 = {
        "Corrupción y seguridad"           : "↑ esperado",
        "Características sociodemográficas": "↑ esperado",
        "Confianza institucional"          : "→ estable",
        "Evaluación económica"             : "→ estable",
        "Percepción política"              : "↑ posible",
    }

    for bloque, row in df_cambio_bloque.iterrows():
        pred = pred_h4.get(bloque, "—")
        direccion = "↑" if row["cambio_absoluto"] > 0 else "↓"
        confirmado = ""
        if pred == "↑ esperado" and row["cambio_absoluto"] > 0:
            confirmado = " ✓ Confirma H4"
        elif pred == "↑ esperado" and row["cambio_absoluto"] <= 0:
            confirmado = " ✗ No confirma H4"
        print(f"  {bloque:<42}: {direccion} ({row['cambio_absoluto']:+.4f})  "
              f"Predicción: {pred}{confirmado}")

print()
print("=" * 60)
print("PRUEBA DE H5")
print("=" * 60)

if 'df_corr' in dir():
    if "test" in df_corr.index and "test" in df_corr.columns and "test" in df_corr.columns:
        r_sp1_sp2 = float(df_corr.loc["test","test"])
        r_sp1_sp3 = float(df_corr.loc["test","test"])
        diferencia = r_sp1_sp2 - r_sp1_sp3

        print(f"  r(test, test) = {r_sp1_sp2:.4f}")
        print(f"  r(test, test) = {r_sp1_sp3:.4f}")
        print(f"  Diferencia  = {diferencia:+.4f}")
        print()
        if r_sp1_sp3 < r_sp1_sp2:
            print("  ✓ H5 CONFIRMADA: la correlación test-test es menor que test-test,")
            print("    indicando cambio estructural post-pandémico en los determinantes.")
        else:
            print("  ✗ H5 NO CONFIRMADA: la correlación test-test no es menor que test-test.")
        if diferencia > 0.15:
            print(f"    La diferencia ({diferencia:.3f}) sugiere cambio estructural sustancial.")
        elif diferencia > 0.05:
            print(f"    La diferencia ({diferencia:.3f}) es moderada.")
        else:
            print(f"    La diferencia ({diferencia:.3f}) es pequeña.")

## 8. Resumen y guardado

In [ ]:
print("=" * 60)
print("RESUMEN — Análisis de estabilidad temporal y regional")
print("=" * 60)
print(f"  Modelo analizado : {MODELO_ESTABILIDAD}")
print(f"  Splits      : test, test, test")
print()
print("Archivos generados:")
for f in sorted(PATHS["FOLDER_RESULTS_FIGURES"].glob("05_*.png")):
    print(f"  {f.name}")
for f in sorted(PATHS["FOLDER_RESULTS_TABLES"].glob("*spearman*")):
    print(f"  {f.name}")
for f in sorted(PATHS["FOLDER_RESULTS_TABLES"].glob("*mae*")):
    print(f"  {f.name}")